# Universal CUDA-JIT phase retrieval
Minimal use of `phase_retrieval_universal_jit.py`.

In [ ]:
import numpy as np
from library import phase_retrieval_universal_jit as pr

In [ ]:
holograms = np.load("data/universal_holograms.npy")
mask_pixel = np.load("data/mask_pixel.npy")
supportmask = np.load("data/supportmask.npy")
state_labels = ["saturated", "saturated", "domains", "domains"]
energy_labels = [778.0, 778.0, 780.0, 780.0]
polarizations = [+1, -1, +1, -1]
illumination_labels = ["beam1", "beam1", "beam1", "beam1"]
saturated_states = {"saturated": +1}

print(pr.cuda_jit_status())
# pr.warm_up_jit(holograms.shape[-2:], modes=["HAPRE", "ER"])

In [ ]:
# The JIT front end uses the same complete recipe as phase_retrieval_universal.py.
recipe = {
    # Update schedule and ordinary phase-retrieval controls.
    "inner_mode": ["HAPRE", "ER"],       # JIT-supported stage algorithms.
    "inner_Nit": [700, 50],               # Iterations in each stage.
    "outer_iterations": 100,              # Update-plus-projection cycles.
    "warmup_mode": ["HAPRE"],            # Independent pre-coupling stages.
    "warmup_Nit": 0,                      # Zero disables warmup.
    "shuffle_observations": True,         # Randomize observation update order.
    "random_seed": None,                  # Randomization seed.
    "beta_zero": 0.5,                     # Beta value(s).
    "beta_mode": "arctan",               # Beta schedule name(s) or arrays.
    "alpha_zero": 0.0,                    # TV strength; active TV currently triggers fallback.
    "alpha_mode": "const",               # Alpha schedule.
    "TV_freq": 1e9,                       # TV update interval.
    "warmup_beta_zero": None,             # None inherits inner setting.
    "warmup_beta_mode": None,             # None inherits inner setting.
    "warmup_alpha_zero": None,            # None inherits inner setting.
    "warmup_alpha_mode": None,            # None inherits inner setting.
    "warmup_TV_freq": None,               # None inherits inner setting.
    "plot_every": 1e9,                    # Error sampling interval.
    "average_img": 1,                     # Best late iterates to average.
    "Fourier_last": True,                 # Final stage Fourier constraint.
    "final_fourier_constraint": True,     # Final measured-amplitude constraint.
    "hologram_intensity_cutoff_vmin": -1, # Percentile baseline subtraction.
    # Universal projection controls.
    "projection_model": "physical_factorized", # physical_factorized, state_energy_beam, none, svd, rank1_spectral.
    "projection_every": 1,                # Projection interval.
    "projection_start": 0,                # First projected outer cycle.
    "projection_relaxation": 1.0,         # Projection blending fraction.
    "projection_constraints_inside_support_only": False, # If True, apply joint projections only inside supportmask.
    "observation_weights": None,          # Positive weight per observation.
    "rank_deficient": "error",           # Flexible-model rank-deficiency behavior.
    "log_floor": 1e-12,                   # Floor before complex log.
    # Physical factorization and spectra.
    "physical_iterations": 20,            # Alternating-fit iterations.
    "saturated_states": saturated_states, # State-to-+1/-1 mapping.
    "charge_spectral_constraint": "free",   # free, kk, known_beta, known_beta_kk.
    "magnetic_spectral_constraint": "free", # Magnetic spectral constraint.
    "energy_values": np.array([778.0, 780.0]), # Unique-energy order.
    "known_charge_beta_spectrum": None,   # Known charge beta(E).
    "known_charge_delta_spectrum": None,  # Known charge delta(E).
    "known_magnetic_beta_spectrum": None, # Known magnetic beta(E).
    "known_magnetic_delta_spectrum": None,# Known magnetic delta(E).
    "charge_absorption_part": "real",    # Charge response absorption location.
    "magnetic_absorption_part": "real",  # Magnetic response absorption location.
    "charge_response_real_range": None,   # Direct Re(q_charge) bounds.
    "charge_response_imag_range": None,   # Direct Im(q_charge) bounds.
    "magnetic_response_real_range": None, # Direct Re(q_magnetic) bounds.
    "magnetic_response_imag_range": None, # Direct Im(q_magnetic) bounds.
    "kk_sign": 1.0,                       # KK sign convention.
    "kk_subtract_baseline": True,         # Remove endpoint baseline.
    "kk_normalize_input": False,          # Normalize KK input.
    "known_spectrum_normalization": "none", # none, maxabs, l2, std.
    "fit_known_spectrum_scale": True,     # Fit physical-spectrum scale.
    "fit_known_spectrum_offset": True,    # Fit physical-spectrum offset.
    # Pure-energy projection options.
    "rank": 1,                            # SVD residual rank.
    "projection_static_mode": "mean",    # mean, first, or none.
    "spectral_constraint": "free",       # Rank-one spectral constraint.
    "known_beta_spectrum": None,          # Rank-one known absorption spectrum.
    "known_delta_spectrum": None,         # Rank-one known dispersion spectrum.
    "absorption_part": "real",           # Rank-one absorption location.
    "known_beta_normalization": "none",  # none, maxabs, l2, std.
    "fit_known_beta_scale": True,         # Fit rank-one scale.
    "fit_known_beta_offset": True,        # Fit rank-one offset.
    # Physical bounds and optional refractive-index conversion.
    "charge_kt_delta_range": None,        # Bounds on k*t*delta_charge.
    "charge_kt_beta_range": None,         # Bounds on k*t*beta_charge.
    "magnetic_kt_delta_range": None,      # Bounds on k*t*delta_magnetic.
    "magnetic_kt_beta_range": None,       # Bounds on k*t*beta_magnetic.
    "wave_numbers": None,                 # k(E) in m^-1 or derive from energy_values.
    "thickness": None,                    # Thickness in metres for index conversion.
    "known_charge_kt_beta_spectrum": None,   # Known k*t*beta_charge(E).
    "known_charge_kt_delta_spectrum": None,  # Known k*t*delta_charge(E).
    "known_magnetic_kt_beta_spectrum": None, # Known k*t*beta_magnetic(E).
    "known_magnetic_kt_delta_spectrum": None,# Known k*t*delta_magnetic(E).
}

fields, components, bsmasks, errors = (
    pr.universal_phase_retrieval_algorithm_jit(
        holograms,
        mask_pixel,
        supportmask,
        state_labels=state_labels,
        energy_labels=energy_labels,
        polarization_coefficients=polarizations,
        illumination_labels=illumination_labels,
        saturated_states=saturated_states,
        universal_recipe=recipe,
        fallback=False,
    )
)